# Módulo 02: Muestreo del Espacio de Parámetros (DoE)
**Curso guiado de GSUA-CSB**

En esta guía interactiva estudiamos las técnicas de **Diseño de Experimentos Computacionales (DoE)** para explorar el hipercubo de parámetros declarados en los modelos dinámicos:
1. **Muestreo Aleatorio Uniforme** (`uniform`).
2. **Hipercubo Latino** (`latin_hypercube` / LHS).
3. **Secuencias de Sobol** (`sobol`).
4. **Muestreo en Escala Logarítmica** (`log_scale`).

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from gsua_csb import design_matrix, UserFunctionModel

# Importar el modelo SIR continuo del Módulo 01
module1_py_path = os.path.abspath(os.path.join("..", "01_Introduccion_Modelos", "python"))
if module1_py_path not in sys.path:
    sys.path.append(module1_py_path)

from sir_continuous_userdefined import create_sir_continuous_model

## 1. Generación de Matrices de Diseño
Construimos el modelo SIR con 4 factores: `['beta', 'gamma', 'S0', 'I0']` y muestreamos $N = 256$ combinaciones con cada método.

In [ ]:
model_sir = create_sir_continuous_model()
N_samples = 256 # Potencia de 2 recomendada para Sobol

M_uni   = design_matrix(model_sir, n=N_samples, method="uniform", seed=42)
M_lhs   = design_matrix(model_sir, n=N_samples, method="latin_hypercube", seed=42)
M_sobol = design_matrix(model_sir, n=N_samples, method="sobol", seed=42)

# Comparación gráfica de proyecciones 2D (beta vs gamma)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].scatter(M_uni[:, 0], M_uni[:, 1], color='tab:red', alpha=0.7, s=18)
axes[0].set_xlabel('beta')
axes[0].set_ylabel('gamma')
axes[0].set_title('Muestreo Uniforme Aleatorio (Aglomeraciones y Huecos)')
axes[0].grid(True)

axes[1].scatter(M_lhs[:, 0], M_lhs[:, 1], color='tab:green', alpha=0.7, s=18)
axes[1].set_xlabel('beta')
axes[1].set_ylabel('gamma')
axes[1].set_title('Hipercubo Latino / LHS (Proyección Unidimensional Estratificada)')
axes[1].grid(True)

axes[2].scatter(M_sobol[:, 0], M_sobol[:, 1], color='tab:blue', alpha=0.7, s=18)
axes[2].set_xlabel('beta')
axes[2].set_ylabel('gamma')
axes[2].set_title('Secuencia de Sobol (Baja Discrepancia Multidimensional)')
axes[2].grid(True)

plt.tight_layout()
plt.show()

## 2. Escala Logarítmica (`log_scale`)
Cuando los rangos de un parámetro abarcan varios órdenes de magnitud (ej. $\beta \in [10^{-6}, 10^{-1}]$), un muestreo lineal concentra casi todas las muestras en la década superior ($0.09$ a $0.10$). Activar `log_scale` realiza el muestreo uniforme en espacio $\log_{10}$.

In [ ]:
def dummy_func(params):
    return params[0] + params[1]

model_log = UserFunctionModel(
    func=dummy_func,
    names=["beta_multidecadal", "gamma_std"],
    range=np.array([[1e-6, 1e-1], [0.05, 0.20]]),
    log_scale=[True, False]
)

M_log = design_matrix(model_log, n=500, method="latin_hypercube", seed=42)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(M_log[:, 0], bins=25, color='tab:orange', edgecolor='black')
axes[0].set_xlabel('beta_multidecadal (Escala Lineal)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución en Valores Reales (Concentrada en década superior)')
axes[0].grid(True)

axes[1].hist(np.log10(M_log[:, 0]), bins=25, color='tab:purple', edgecolor='black')
axes[1].set_xlabel('log10(beta_multidecadal) (Escala Logarítmica)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución en log10 (Uniforme a través de todas las décadas)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 3. Preguntas de Análisis para el Estudiante

1. **Discrepancia**: Al inspeccionar los gráficos de dispersión 2D de `beta` vs `gamma`, ¿qué método produce un llenado de espacio más homogéneo y libre de agrupamientos accidentales?
2. **Propiedad de LHS**: Explique por qué dividir cada dimensión en $N$ intervalos de igual probabilidad garantiza una mejor representación unidimensional de cada parámetro en comparación con el muestreo aleatorio uniforme puro.
3. **Importancia de `log_scale`**: Suponga que el verdadero valor de $\beta$ es $2 \times 10^{-5}$ y se muestrea en el rango $[10^{-6}, 10^{-1}]$. ¿Cuántas muestras caerían cerca del valor real si se usa muestreo lineal vs. muestreo logarítmico?